Explore hyper-parameters

Training YOLOv8n with Real Waste images in Colab.
This will capture training time.

In [ ]:
import os
print(os.getcwd())

/content


Because Colab uses temporarly environments, we need to install ultralytics every time I need to downlaod a model and train it.

In [ ]:
import os
print(os.listdir('/content'))

['.config', 'sample_data']


Mount google drive in this environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.0 MB/s eta 0:00:00


Check if we have GPU working.

In [1]:
import torch
print(torch.cuda.is_available())
# Should return: True

False


Create data.yaml

In [ ]:
#0: Cardboard
#1: Food Organics
#2: Glass
#3: Metal
#4: Miscellaneous Trash
#5: Paper
#6: Plastic
#7: Textile Trash
#8: Vegetation

import yaml

# The path to your data.yaml
yaml_path = 'realwaste-yolo/RealWaste/data.yaml'### in google drive'/content/drive/MyDrive/Colab Notebooks/RealWaste/data.yaml'

# The actual root folder where your images live
dataset_root = 'realwaste-yolo/RealWaste/'### in google drive'/content/drive/MyDrive/Colab Notebooks/RealWaste/'

data = {
    'path': dataset_root, # This is the base path
    'train': 'images/train', # Relative to the path above
    'val': 'images/val',   # Change to 'val/images' if your folder is named 'val'
    'test': 'images/test',   # (Optional)

    'nc': 9, # Make sure this matches your number of classes
    'names': ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
}

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"Successfully updated {yaml_path}")

Successfully updated /content/drive/MyDrive/Colab Notebooks/RealWaste/data.yaml


Training Block

In [6]:
from ultralytics import YOLO
import torch
from datetime import datetime

# Start time
start_time = datetime.now()
print("Training started at:", start_time.strftime("%Y-%m-%d %H:%M:%S"))
# ======> Train the model
def main():

    # 1. Check for NVIDIA GPU (Colab/PC)
    # 2. Check for Apple Silicon (Mac)
    # 3. Default to CPU
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

    print(f"I am training on: {device.upper()}")

    ### Smaller model.
    model = YOLO("yolov8n.pt")

    model.train(
        data="realwaste-yolo/RealWaste/data.yaml",
        ### '/content/RealWaste/data.yaml'
        ### data="/content/drive/MyDrive/Colab Notebooks/RealWaste/data.yaml",
        epochs=100, ###==> Hyper-parameters
        lr0=0.005,
        lrf=0.1,
        momentum=0.9,
        weight_decay=0.0001,
        ###warmup_epochs=5,
        ###warmup_momentum=0.8,
        ###warmup_bias_lr=0.1, ###<==
        imgsz=640,
        batch=16,
        patience=25,
        device=device,
        workers=0,  # safer in Jupyter if needed
        project="runs",
        name="yolov8_pytorch_first"
    )

if __name__ == "__main__":
    main()
# <=====

# End time
end_time = datetime.now()
print("Training ended at:", end_time.strftime("%Y-%m-%d %H:%M:%S"))

# Elapsed time
elapsed = end_time - start_time

# Format nicely (hours, minutes, seconds)
hours, remainder = divmod(elapsed.total_seconds(), 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Elapsed time: {int(hours)}h {int(minutes)}m {int(seconds)}s")

Training started at: 2026-05-05 14:10:26
I am training on: MPS
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.41 🚀 Python-3.10.20 torch-2.11.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=realwaste-yolo/RealWaste/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, mo